# Accepted Loan HistGradientBoosting Modeling And Tuning

This notebook trains baseline candidates and tuned candidates for `hist_gradient_boosting` using the chronological baseline preprocessing exports; candidate search may use a stratified train-period sample for runtime. Thresholds are selected on validation only.

## 1. Setup

In [1]:
MODEL_FAMILY = 'hist_gradient_boosting'
MODEL_LABEL = 'HistGradientBoostingClassifier'

from __future__ import annotations

import json
import os
import time
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
TARGET_PRECISION = 0.40
REVIEW_RATES = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Cleaning").exists() and (candidate / "Modeling").exists():
            return candidate
    raise FileNotFoundError("Could not find CreditRiskRAG project root")

PROJECT_ROOT = find_project_root()
PREPROCESSING_DATASET_DIR = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "datasets"
MODELING_OUTPUT_ROOT = PROJECT_ROOT / "Modeling" / "modeling_outputs"
MODEL_OUTPUT_ROOT = MODELING_OUTPUT_ROOT / MODEL_FAMILY
TABLE_DIR = MODEL_OUTPUT_ROOT / "tables"
PLOT_DIR = MODEL_OUTPUT_ROOT / "plots"
MODEL_DIR = MODEL_OUTPUT_ROOT / "models"
for directory in [TABLE_DIR, PLOT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Preprocessing datasets:", PREPROCESSING_DATASET_DIR)
print("Model outputs:", MODEL_OUTPUT_ROOT)

from sklearn.ensemble import HistGradientBoostingClassifier

Project root: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Preprocessing datasets: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/Preprocessing/preprocessing_outputs/datasets
Model outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting


## 2. Load Preprocessed Baseline Data

In [2]:

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"{MODEL_FAMILY}_{name}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def save_plot(fig, name: str) -> Path:
    path = PLOT_DIR / f"{MODEL_FAMILY}_{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)
    return path

def load_parquet(name: str) -> pd.DataFrame:
    path = PREPROCESSING_DATASET_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path)

X_train = load_parquet("baseline_train_X")
X_validation = load_parquet("baseline_validation_X")
X_test = load_parquet("baseline_test_X")
y_train = load_parquet("train_y")["target_bad"].astype(int)
y_validation = load_parquet("validation_y")["target_bad"].astype(int)
y_test = load_parquet("test_y")["target_bad"].astype(int)

input_summary = pd.DataFrame([
    {"split": "train", "rows": len(X_train), "columns": X_train.shape[1], "bad_rate": y_train.mean()},
    {"split": "validation", "rows": len(X_validation), "columns": X_validation.shape[1], "bad_rate": y_validation.mean()},
    {"split": "test", "rows": len(X_test), "columns": X_test.shape[1], "bad_rate": y_test.mean()},
])
input_summary["bad_rate"] = input_summary["bad_rate"].round(6)
save_table(input_summary, "input_summary")
display(input_summary)


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/tables/hist_gradient_boosting_input_summary.csv


,split,rows,columns,bad_rate
0,train,962641,100,0.188300
1,validation,186920,100,0.246763
2,test,195749,100,0.210315


## 3. Evaluation Helpers

In [3]:

def predict_positive_probability(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "n_jobs"):
        try:
            model.n_jobs = 1
        except Exception:
            pass
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    score = model.decision_function(X)
    return 1.0 / (1.0 + np.exp(-score))

def threshold_for_best_f1(y_true: pd.Series, y_score: np.ndarray) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0
    f1 = (2 * precision[:-1] * recall[:-1]) / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx]), float(f1[idx]), float(precision[idx]), float(recall[idx])

def threshold_for_target_precision(y_true: pd.Series, y_score: np.ndarray, target_precision: float) -> tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0, 0.0, 0.0
    candidate = np.where(precision[:-1] >= target_precision)[0]
    if len(candidate) == 0:
        idx = int(np.nanargmax(precision[:-1]))
    else:
        idx = int(candidate[np.nanargmax(recall[:-1][candidate])])
    f1 = (2 * precision[idx] * recall[idx]) / max(precision[idx] + recall[idx], 1e-12)
    return float(thresholds[idx]), float(f1), float(precision[idx]), float(recall[idx])

def evaluate_at_threshold(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray, threshold: float, operating_point: str) -> dict:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model_family": MODEL_FAMILY,
        "model": model_name,
        "split": split,
        "operating_point": operating_point,
        "rows": len(y_true),
        "bad_rate": round(float(y_true.mean()), 6),
        "threshold": round(float(threshold), 6),
        "roc_auc": round(float(roc_auc_score(y_true, y_score)), 6),
        "pr_auc": round(float(average_precision_score(y_true, y_score)), 6),
        "brier_score": round(float(brier_score_loss(y_true, y_score)), 6),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 6),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 6),
        "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 6),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def review_volume_metrics(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray) -> pd.DataFrame:
    order = np.argsort(-y_score)
    y_sorted = np.asarray(y_true)[order]
    base_bad_rate = float(np.mean(y_sorted))
    total_bad = int(y_sorted.sum())
    rows = []
    for rate in REVIEW_RATES:
        review_count = max(1, int(np.ceil(len(y_sorted) * rate)))
        reviewed = y_sorted[:review_count]
        captured_bad = int(reviewed.sum())
        precision = captured_bad / review_count
        recall = captured_bad / total_bad if total_bad else np.nan
        rows.append({
            "model_family": MODEL_FAMILY,
            "model": model_name,
            "split": split,
            "review_pct": round(rate * 100, 2),
            "review_count": int(review_count),
            "captured_bad": captured_bad,
            "precision": round(float(precision), 6),
            "recall": round(float(recall), 6),
            "base_bad_rate": round(base_bad_rate, 6),
            "lift_over_base_bad_rate": round(float(precision / base_bad_rate), 6) if base_bad_rate else np.nan,
        })
    return pd.DataFrame(rows)

def fit_candidates(candidates: list[dict], sample_rows: int | None = None) -> tuple[pd.DataFrame, dict]:
    if sample_rows and len(X_train) > sample_rows:
        sample_idx = y_train.groupby(y_train).sample(frac=sample_rows / len(y_train), random_state=RANDOM_STATE).index
        X_fit = X_train.loc[sample_idx]
        y_fit = y_train.loc[sample_idx]
    else:
        X_fit = X_train
        y_fit = y_train

    fitted_models = {}
    rows = []
    for candidate in candidates:
        name = candidate["candidate"]
        params = candidate["params"]
        print(f"Training {name}: {params}")
        start = time.perf_counter()
        model = build_model(params)
        fit_kwargs = build_fit_kwargs(y_fit)
        model.fit(X_fit, y_fit, **fit_kwargs)
        seconds = time.perf_counter() - start

        validation_score = predict_positive_probability(model, X_validation)
        best_threshold, best_f1, best_precision, best_recall = threshold_for_best_f1(y_validation, validation_score)
        precision_threshold, precision_f1, precision_value, precision_recall = threshold_for_target_precision(
            y_validation, validation_score, TARGET_PRECISION
        )
        row = {
            "model_family": MODEL_FAMILY,
            "candidate": name,
            "params": json.dumps(params, sort_keys=True),
            "fit_rows": len(X_fit),
            "fit_bad_rate": round(float(y_fit.mean()), 6),
            "fit_seconds": round(float(seconds), 3),
            "roc_auc": round(float(roc_auc_score(y_validation, validation_score)), 6),
            "pr_auc": round(float(average_precision_score(y_validation, validation_score)), 6),
            "best_f1_threshold": round(best_threshold, 6),
            "best_f1": round(best_f1, 6),
            "best_f1_precision": round(best_precision, 6),
            "best_f1_recall": round(best_recall, 6),
            "target_precision_threshold": round(precision_threshold, 6),
            "target_precision_f1": round(precision_f1, 6),
            "target_precision": round(precision_value, 6),
            "target_precision_recall": round(precision_recall, 6),
        }
        rows.append(row)
        fitted_models[name] = model
        print(f"Finished {name}: best_f1={best_f1:.4f}, precision={best_precision:.4f}, recall={best_recall:.4f}, seconds={seconds:.1f}")
    results = pd.DataFrame(rows).sort_values(["best_f1", "best_f1_precision", "pr_auc"], ascending=False)
    return results, fitted_models

def evaluate_selected_model(candidate_row: pd.Series, model) -> pd.DataFrame:
    scores = {
        "train": predict_positive_probability(model, X_train),
        "validation": predict_positive_probability(model, X_validation),
        "test": predict_positive_probability(model, X_test),
    }
    y_parts = {"train": y_train, "validation": y_validation, "test": y_test}
    rows = []
    for split, y_part in y_parts.items():
        rows.append(evaluate_at_threshold(candidate_row.candidate, split, y_part, scores[split], candidate_row.best_f1_threshold, "best_validation_f1"))
        rows.append(evaluate_at_threshold(candidate_row.candidate, split, y_part, scores[split], candidate_row.target_precision_threshold, "target_validation_precision"))
    review_rows = [review_volume_metrics(candidate_row.candidate, split, y_parts[split], scores[split]) for split in ["validation", "test"]]
    review_df = pd.concat(review_rows, ignore_index=True)
    return pd.DataFrame(rows), review_df


## 4. Candidate Grid

In [4]:

def build_model(params: dict) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        learning_rate=params["learning_rate"],
        max_iter=params["max_iter"],
        max_leaf_nodes=params["max_leaf_nodes"],
        l2_regularization=params["l2_regularization"],
        early_stopping=True,
        validation_fraction=0.10,
        random_state=RANDOM_STATE,
    )

def build_fit_kwargs(y_fit: pd.Series) -> dict:
    return {"sample_weight": compute_sample_weight(class_weight="balanced", y=y_fit)}

CANDIDATES = [
    {"candidate": "hist_gradient_boosting_01", "params": {"learning_rate": 0.04, "max_iter": 180, "max_leaf_nodes": 31, "l2_regularization": 0.0}},
    {"candidate": "hist_gradient_boosting_02", "params": {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 31, "l2_regularization": 0.0}},
    {"candidate": "hist_gradient_boosting_03", "params": {"learning_rate": 0.05, "max_iter": 220, "max_leaf_nodes": 45, "l2_regularization": 0.01}},
    {"candidate": "hist_gradient_boosting_04", "params": {"learning_rate": 0.06, "max_iter": 220, "max_leaf_nodes": 45, "l2_regularization": 0.0}},
    {"candidate": "hist_gradient_boosting_05", "params": {"learning_rate": 0.07, "max_iter": 180, "max_leaf_nodes": 45, "l2_regularization": 0.01}},
    {"candidate": "hist_gradient_boosting_06", "params": {"learning_rate": 0.04, "max_iter": 260, "max_leaf_nodes": 63, "l2_regularization": 0.01}},
]
FIT_SAMPLE_ROWS = 300_000


## 5. Train, Tune, And Evaluate

In [5]:

candidate_results, fitted_models = fit_candidates(CANDIDATES, sample_rows=FIT_SAMPLE_ROWS)
save_table(candidate_results, "candidate_results")
display(candidate_results)

winner = candidate_results.iloc[0]
selected_model = fitted_models[winner.candidate]
selected_metrics, review_volume_precision = evaluate_selected_model(winner, selected_model)
save_table(pd.DataFrame([winner]), "selected_candidate")
save_table(selected_metrics, "selected_model_metrics")
save_table(review_volume_precision, "review_volume_precision")
display(selected_metrics)
display(review_volume_precision)

model_path = MODEL_DIR / f"{MODEL_FAMILY}_selected_model.joblib"
joblib.dump(selected_model, model_path)
artifact_table = pd.DataFrame([{
    "model_family": MODEL_FAMILY,
    "candidate": winner.candidate,
    "artifact_path": str(model_path),
}])
save_table(artifact_table, "model_artifact")
print("Saved:", model_path)

for metric in ["f1", "precision", "recall", "pr_auc", "roc_auc"]:
    plot_df = selected_metrics[(selected_metrics["split"].isin(["validation", "test"])) & (selected_metrics["operating_point"] == "best_validation_f1")]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(plot_df["split"], plot_df[metric])
    ax.set_title(f"{MODEL_LABEL} {metric.upper()} by split")
    ax.set_ylim(0, max(plot_df[metric].max() * 1.15, 0.05))
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, f"selected_{metric}_comparison")

fig, ax = plt.subplots(figsize=(8, 4.5))
for split, group in review_volume_precision.groupby("split"):
    ax.plot(group["review_pct"], group["precision"], marker="o", label=split)
ax.set_title(f"{MODEL_LABEL} precision at fixed review volumes")
ax.set_xlabel("Reviewed applications (%)")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
ax.legend()
save_plot(fig, "precision_by_review_volume")


Training hist_gradient_boosting_01: {'learning_rate': 0.04, 'max_iter': 180, 'max_leaf_nodes': 31, 'l2_regularization': 0.0}


Finished hist_gradient_boosting_01: best_f1=0.4734, precision=0.3564, recall=0.7044, seconds=34.0
Training hist_gradient_boosting_02: {'learning_rate': 0.05, 'max_iter': 200, 'max_leaf_nodes': 31, 'l2_regularization': 0.0}


Finished hist_gradient_boosting_02: best_f1=0.4736, precision=0.3582, recall=0.6988, seconds=29.7
Training hist_gradient_boosting_03: {'learning_rate': 0.05, 'max_iter': 220, 'max_leaf_nodes': 45, 'l2_regularization': 0.01}


Finished hist_gradient_boosting_03: best_f1=0.4739, precision=0.3612, recall=0.6891, seconds=29.8
Training hist_gradient_boosting_04: {'learning_rate': 0.06, 'max_iter': 220, 'max_leaf_nodes': 45, 'l2_regularization': 0.0}


Finished hist_gradient_boosting_04: best_f1=0.4743, precision=0.3595, recall=0.6967, seconds=25.0
Training hist_gradient_boosting_05: {'learning_rate': 0.07, 'max_iter': 180, 'max_leaf_nodes': 45, 'l2_regularization': 0.01}


...

,model_family,model,split,operating_point,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,hist_gradient_boosting,hist_gradient_boosting_04,train,best_validation_f1,962641,0.188300,0.472937,0.734187,0.393537,0.207696,0.303810,0.723388,0.427907,480899,300477,50140,131125
1,hist_gradient_boosting,hist_gradient_boosting_04,train,target_validation_precision,962641,0.188300,0.550043,0.734187,0.393537,0.207696,0.348096,0.582495,0.435775,583638,197738,75679,105586
2,hist_gradient_boosting,hist_gradient_boosting_04,validation,best_validation_f1,186920,0.246763,0.472937,0.700829,0.423500,0.217717,0.359479,0.696715,0.474259,83535,57260,13989,32136
3,hist_gradient_boosting,hist_gradient_boosting_04,validation,target_validation_precision,186920,0.246763,0.550043,0.700829,0.423500,0.217717,0.400006,0.538320,0.458969,103551,37244,21295,24830
4,hist_gradient_boosting,hist_gradient_boosting_04,test,best_validation_f1,195749,0.210315,0.472937,0.708493,0.377185,0.212358,0.319695,0.693143,0.437571,93856,60724,12633,28536
5,hist_gradient_boosting,hist_gradient_boosting_04,test,target_validation_precision,195749,0.210315,0.550043,0.708493,0.377185,0.212358,0.357576,0.543856,0.431469,114354,40226,18779,22390


,model_family,model,split,review_pct,review_count,captured_bad,precision,recall,base_bad_rate,lift_over_base_bad_rate
0,hist_gradient_boosting,hist_gradient_boosting_04,validation,1.0,1870,1208,0.645989,0.026190,0.246763,2.617850
1,hist_gradient_boosting,hist_gradient_boosting_04,validation,2.0,3739,2293,0.613266,0.049713,0.246763,2.485238
2,hist_gradient_boosting,hist_gradient_boosting_04,validation,5.0,9346,5171,0.553285,0.112108,0.246763,2.242168
3,hist_gradient_boosting,hist_gradient_boosting_04,validation,10.0,18692,9525,0.509576,0.206504,0.246763,2.065041
4,hist_gradient_boosting,hist_gradient_boosting_04,validation,15.0,28038,13311,0.474749,0.288585,0.246763,1.923902
5,hist_gradient_boosting,hist_gradient_boosting_04,validation,20.0,37384,16752,0.448106,0.363187,0.246763,1.815935
6,hist_gradient_boosting,hist_gradient_boosting_04,validation,25.0,46730,19998,0.427948,0.433561,0.246763,1.734244
7,hist_gradient_boosting,hist_gradient_boosting_04,validation,30.0,56076,23030,0.410693,0.499295,0.246763,1.664318
8,hist_gradient_boosting,hist_gradient_boosting_04,test,1.0,1958,1092,0.557712,0.026525,0.210315,2.651790
9,hist_gradient_boosting,hist_gradient_boosting_04,test,2.0,3915,2103,0.537165,0.051082,0.210315,2.554093


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/tables/hist_gradient_boosting_model_artifact.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/models/hist_gradient_boosting_selected_model.joblib
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/plots/hist_gradient_boosting_selected_f1_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/plots/hist_gradient_boosting_selected_precision_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/plots/hist_gradient_boosting_precision_by_review_volume.png


PosixPath('/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/hist_gradient_boosting/plots/hist_gradient_boosting_precision_by_review_volume.png')

## 6. Confusion Matrix And Per-Class Metrics

Show the confusion-matrix layout for each split and operating point. Class `0` is `Fully Paid`; class `1` is `Charged Off`. Precision, recall, and F1 are also reported separately for each class.

In [ ]:
def build_confusion_matrix_tables(metrics: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    matrix_rows = []
    per_class_rows = []

    def safe_div(num: float, den: float) -> float:
        return float(num / den) if den else 0.0

    for _, row in metrics.iterrows():
        tn = int(row["tn"])
        fp = int(row["fp"])
        fn = int(row["fn"])
        tp = int(row["tp"])
        base = {
            "model_family": MODEL_FAMILY,
            "candidate": row["model"],
            "split": row["split"],
            "operating_point": row["operating_point"],
            "threshold": row["threshold"],
        }

        matrix_rows.extend([
            {**base, "actual_label": 0, "actual_class": "Fully Paid", "predicted_label": 0, "predicted_class": "Fully Paid", "count": tn, "cell": "TN"},
            {**base, "actual_label": 0, "actual_class": "Fully Paid", "predicted_label": 1, "predicted_class": "Charged Off", "count": fp, "cell": "FP"},
            {**base, "actual_label": 1, "actual_class": "Charged Off", "predicted_label": 0, "predicted_class": "Fully Paid", "count": fn, "cell": "FN"},
            {**base, "actual_label": 1, "actual_class": "Charged Off", "predicted_label": 1, "predicted_class": "Charged Off", "count": tp, "cell": "TP"},
        ])

        precision_0 = safe_div(tn, tn + fn)
        recall_0 = safe_div(tn, tn + fp)
        f1_0 = safe_div(2 * precision_0 * recall_0, precision_0 + recall_0)
        precision_1 = safe_div(tp, tp + fp)
        recall_1 = safe_div(tp, tp + fn)
        f1_1 = safe_div(2 * precision_1 * recall_1, precision_1 + recall_1)

        per_class_rows.extend([
            {**base, "class_label": 0, "class_name": "Fully Paid", "precision": round(precision_0, 6), "recall": round(recall_0, 6), "f1": round(f1_0, 6), "support": tn + fp},
            {**base, "class_label": 1, "class_name": "Charged Off", "precision": round(precision_1, 6), "recall": round(recall_1, 6), "f1": round(f1_1, 6), "support": tp + fn},
        ])

    return pd.DataFrame(matrix_rows), pd.DataFrame(per_class_rows)


if "selected_metrics" not in globals():
    selected_metrics_path = TABLE_DIR / f"{MODEL_FAMILY}_selected_model_metrics.csv"
    if not selected_metrics_path.exists():
        raise FileNotFoundError(f"Missing {selected_metrics_path}. Run the training/evaluation section first.")
    selected_metrics = pd.read_csv(selected_metrics_path)

confusion_matrix_long, per_class_metrics = build_confusion_matrix_tables(selected_metrics)
save_table(confusion_matrix_long, "confusion_matrix")
save_table(per_class_metrics, "per_class_metrics")

best_f1_confusion_matrix = confusion_matrix_long[
    (confusion_matrix_long["split"].isin(["validation", "test"]))
    & (confusion_matrix_long["operating_point"] == "best_validation_f1")
]
best_f1_per_class_metrics = per_class_metrics[
    (per_class_metrics["split"].isin(["validation", "test"]))
    & (per_class_metrics["operating_point"] == "best_validation_f1")
]

display(best_f1_confusion_matrix)
display(best_f1_per_class_metrics)

## 7. Notes

Use validation metrics for model/threshold selection. Use test metrics only for final reporting after selection.